# RG3 gas history-window comparison
Same four sensors and model; 10, 100, 300 past observations. No future/current-shot history.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, f1_score
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
tables=root/'reports/tables'
# Reuse audited raw history preparation, stopping before feature/report generation.
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
prep=''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source'])
exec(prep.split('for feature in features:')[0])
sensors=['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']
windows=[10,100,300]
for f in sensors:
    previous=observed.groupby('history_segment')[f].shift()
    for w in windows:
        observed[f+'__std'+str(w)]=previous.groupby(observed.history_segment).transform(lambda s:s.rolling(w,min_periods=w).std())
        observed[f+'__count'+str(w)]=previous.groupby(observed.history_segment).transform(lambda s:s.rolling(w,min_periods=1).count())
shots=label_shots.merge(observed,on=keys,how='left',validate='one_to_one')
data=shots.loc[shots['product'].eq('RG3') & (shots.PassOrFail.eq(0)|shots.Reason.str.contains('가스',regex=False))].reset_index(drop=True)
y=data.Reason.str.contains('가스',regex=False).astype(int)
assert len(data)==583 and y.sum()==17
coverage=[]
for w in windows:
    cols=[f+'__std'+str(w) for f in sensors]
    good=data[cols].notna().all(axis=1)
    coverage.append(dict(window=w,shots=len(data),complete_shots=int(good.sum()),complete_gas=int(y[good].sum()),missing_shots=int((~good).sum()),missing_gas=int(y[~good].sum())))
coverage=pd.DataFrame(coverage)
print('COVERAGE\n'+coverage.to_string(index=False),flush=True)
# Longest history available: common cohort removes unequal missingness across windows.
common=data[[f+'__std300' for f in sensors]].notna().all(axis=1)
results=[]; predictions=[]; byblock=[]
def evaluate(frame,target,cols,window,cohort):
    groups=frame.comparison_block
    probs=np.full(len(frame),np.nan)
    for train,test in LeaveOneGroupOut().split(frame,target,groups):
        assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
        model=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1))
        model.fit(frame.iloc[train][cols],target.iloc[train])
        classes=model[-1].classes_
        probs[test]=model.predict_proba(frame.iloc[test][cols])[:,list(classes).index(1)] if 1 in classes else 0
    assert np.isfinite(probs).all()
    pred=probs>=0.5
    tn,fp,fn,tp=confusion_matrix(target,pred,labels=[0,1]).ravel()
    tie=np.random.RandomState(42).random_sample(len(frame)); selected=np.zeros(len(frame),dtype=bool)
    for block in groups.unique():
        idx=np.flatnonzero(groups.eq(block).to_numpy())
        order=idx[np.lexsort((tie[idx],-probs[idx]))]
        selected[order[:int(np.ceil(len(idx)*0.1))]]=True
        byblock.append(dict(window=window,cohort=cohort,block=block,shots=len(idx),gas=int(target.iloc[idx].sum()),caught_top10=int((selected[idx]&target.iloc[idx].astype(bool)).sum())))
    row=dict(window=window,cohort=cohort,shots=len(frame),gas=int(target.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=precision_score(target,pred,zero_division=0),recall=recall_score(target,pred,zero_division=0),F1=f1_score(target,pred,zero_division=0),AP=average_precision_score(target,probs),selected_top10=int(selected.sum()),caught_top10=int((selected&target.astype(bool)).sum()))
    results.append(row)
    out=frame[keys+['comparison_block','Reason']].copy()
    out['gas']=target.to_numpy();out['probability']=probs;out['selected_top10']=selected;out['window']=window;out['cohort']=cohort
    predictions.append(out)
    print(row,flush=True)
# Exact former baseline (10-shot window, at least 5 valid values), for reconciliation.
oldshots=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
oldrg=oldshots.loc[oldshots['product'].eq('RG3') & (oldshots.PassOrFail.eq(0)|oldshots.Reason.str.contains('가스',regex=False))].reset_index(drop=True)
evaluate(oldrg,oldrg.Reason.str.contains('가스',regex=False).astype(int),[f+'__past10_std' for f in sensors],10,'previous_min5')
for w in windows:
    cols=[f+'__std'+str(w) for f in sensors]
    evaluate(data,y,cols,w,'all_with_train_imputation')
    if common.sum()>0 and y[common].sum()>0 and data.loc[common,'comparison_block'].nunique()>1:
        evaluate(data.loc[common].reset_index(drop=True),y.loc[common].reset_index(drop=True),cols,w,'common_complete300')
metrics=pd.DataFrame(results)
metrics.to_csv(tables/'rg3_window_comparison_metrics.csv',index=False,encoding='utf-8-sig')
coverage.to_csv(tables/'rg3_window_history_coverage.csv',index=False,encoding='utf-8-sig')
pd.concat(predictions).to_csv(tables/'rg3_window_oof_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(byblock).to_csv(tables/'rg3_window_metrics_by_block.csv',index=False,encoding='utf-8-sig')
# Observational overlap: evaluate gas exceedance of normal 90th percentile in each block.
overlap=[]
for w in windows:
    for block,g in data.groupby('comparison_block'):
        target=g.Reason.str.contains('가스',regex=False)
        for f in sensors:
            col=f+'__std'+str(w)
            normal=g.loc[~target,col].dropna();bad=g.loc[target,col].dropna()
            q90=normal.quantile(.9) if len(normal) else np.nan
            overlap.append(dict(window=w,block=block,sensor=f,normal_n=len(normal),gas_n=len(bad),normal_median=normal.median(),gas_median=bad.median(),normal_q90=q90,gas_above_normal_q90=int(bad.gt(q90).sum())))
pd.DataFrame(overlap).to_csv(tables/'rg3_window_sensor_overlap.csv',index=False,encoding='utf-8-sig')
lines=['# RG3 가스: 이전 10·100·300 Shot 비교','',
'정상 566 Shot과 가스 17 Shot, 센서 4개(배럴2·호퍼·쿠션·배럴5)의 과거 표준편차만 사용했다. 미성형 8 Shot은 제외했다.',
'','이전 10개는 짧은 변화 관찰용 초기 설정이며 최적값으로 검증하지 않았다. 100·300개는 더 긴 공정 상태를 반영하지만 짧은 이상을 희석할 수도 있다.',
'','현재 Shot을 제외하고 비라벨 센서 이력을 연결했다. 동일 설비에서 제품 전환 또는 30분 초과 공백이면 이력을 초기화했다. 실제 Run ID는 아니다. 100·300은 경과 시간이 아니라 기록된 고유 Shot 개수다.',
'','공통 완전이력 비교는 네 센서 모두 이전 300개 유효값이 있는 동일 Shot만 사용한다. 전체 비교에서는 각 창 길이를 모두 채우지 못하면 결측 처리하고 학습 구간 중앙값으로 대체한다. 이전 기준은 10개 중 최소5개를 사용했으므로 별도로 재현했다.',
'','같은 Random Forest(200 트리, 깊이8, 최소 말단3, 균형 가중치, 시드42), 관측 구간 전체 제외 평가. 임계값0.5 고정. 상위 약10%는 각 평가 구간에서 선정. 미래 구간이 학습에 포함될 수 있으므로 실제 미래 예측 시험과 다르다.',
'','## 이력 충족','', '| 이전 Shot 수 | 완전이력 Shot | 완전이력 가스 | 이력 부족 Shot | 이력 부족 가스 |','|---|---:|---:|---:|---:|']
for r in coverage.itertuples(): lines.append(f'| {r.window} | {r.complete_shots}/583 | {r.complete_gas}/17 | {r.missing_shots} | {r.missing_gas} |')
lines+=['','## 성능','', '| 평가 대상 | 이전 Shot 수 | 전체/가스 | TP | FP | FN | TN | AP | 상위10% 발견/가스 | 검사 수 |','|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|']
names={'previous_min5':'이전 기준(최소5개)','all_with_train_imputation':'전체(이력 부족 대체)','common_complete300':'동일 완전이력 Shot'}
for r in metrics.itertuples(): lines.append(f'| {names[r.cohort]} | {r.window} | {r.shots}/{r.gas} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.AP:.4f} | {r.caught_top10}/{r.gas} | {r.selected_top10} |')
lines+=['','## 해석','',
'긴 창이 우수한지는 같은 평가 Shot과 불량 수로 비교해야 한다. 불량은17개뿐이고 창 길이 선택도 같은 평가 데이터에 근거하므로 독립 검증이 필요하다. 표준편차가 높다는 사실은 원인이나 가스 판별 규칙을 뜻하지 않는다.',
'','구간별 정상90백분위 초과 가스 수는 별도 센서 비교표에 저장했다. 이 기준은 정상 분포와의 겹침을 설명하는 용도이며 모델 입력·임계값 선정에는 사용하지 않았다.']
(root/'reports/rg3_window_comparison.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('DONE',flush=True)



COVERAGE
 window  shots  complete_shots  complete_gas  missing_shots  missing_gas
     10    583             573            16             10            1
    100    583             497            14             86            3
    300    583             336            10            247            7


{'window': 10, 'cohort': 'previous_min5', 'shots': 583, 'gas': 17, 'TP': 1, 'FP': 6, 'FN': 16, 'TN': 560, 'precision': 0.14285714285714285, 'recall': 0.058823529411764705, 'F1': 0.08333333333333333, 'AP': 0.13871113959502832, 'selected_top10': 62, 'caught_top10': 6}


{'window': 10, 'cohort': 'all_with_train_imputation', 'shots': 583, 'gas': 17, 'TP': 1, 'FP': 3, 'FN': 16, 'TN': 563, 'precision': 0.25, 'recall': 0.058823529411764705, 'F1': 0.09523809523809523, 'AP': 0.10320152374311203, 'selected_top10': 62, 'caught_top10': 5}


{'window': 10, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 0, 'FP': 5, 'FN': 10, 'TN': 321, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'AP': 0.08970185007693794, 'selected_top10': 36, 'caught_top10': 3}


{'window': 100, 'cohort': 'all_with_train_imputation', 'shots': 583, 'gas': 17, 'TP': 0, 'FP': 13, 'FN': 17, 'TN': 553, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'AP': 0.035292645387605584, 'selected_top10': 62, 'caught_top10': 3}


{'window': 100, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 0, 'FP': 0, 'FN': 10, 'TN': 326, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'AP': 0.09639062968442835, 'selected_top10': 36, 'caught_top10': 3}


{'window': 300, 'cohort': 'all_with_train_imputation', 'shots': 583, 'gas': 17, 'TP': 4, 'FP': 170, 'FN': 13, 'TN': 396, 'precision': 0.022988505747126436, 'recall': 0.23529411764705882, 'F1': 0.041884816753926704, 'AP': 0.02889650861115529, 'selected_top10': 62, 'caught_top10': 2}


{'window': 300, 'cohort': 'common_complete300', 'shots': 336, 'gas': 10, 'TP': 0, 'FP': 0, 'FN': 10, 'TN': 326, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'AP': 0.03221217177483095, 'selected_top10': 36, 'caught_top10': 2}


DONE
